In [ ]:
from __future__ import annotations
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
PROJECT_ROOT  = Path("/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral")
OUT_DIR       = Path(".")
ARTIFACT_ROOT = OUT_DIR / "dl_artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES         = ["0_control", "1_significant_mr", "2_significant_ms"]
FLAT_LABEL_TO_CLASS = {"Kontrol": 0, "MR": 1, "MS": 2}
LEAD_ORDER = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_FOLDS  = 5

# Per-model configuration
MODELS_CONFIG = {
    "rib_ext": {
        "artifact_dir": "ribeiro_resnet_ecg_features_ext_tiffany_flat",
        "arch": "rib_with_intervals",
        "in_channels": 36,
        "n_interval": 12,
        "intervals_csv": PROJECT_ROOT / "ecg_features_extended.csv",
        "interval_cols": ["rr_ms","rr_std_ms","p_dur_ms","qrs_dur_ms","qtc_ms","sokolow",
                           "sv1_amp","rv5_amp","t_amp_V4_uv","t_amp_V5_uv","s_amp_aVF_uv","p_amp_V1_uv"],
    },
    "rib_ptbxl_iv": {
        "artifact_dir": "ribeiro_resnet_ecg_intervals_tiffany_flat",
        "arch": "rib_with_intervals",
        "in_channels": 36,
        "n_interval": 6,
        "intervals_csv": PROJECT_ROOT / "ecg_intervals.csv",
        "interval_cols": ["rr_ms","rr_std_ms","p_dur_ms","qrs_dur_ms","qtc_ms","sokolow"],
    },
    "incep_iv": {
        "artifact_dir": "inceptiontime_ecg_intervals_tiffany_flat_nf64",
        "arch": "incep_with_intervals",
        "in_channels": 12,
        "n_interval": 6,
        "intervals_csv": PROJECT_ROOT / "ecg_intervals.csv",
        "interval_cols": ["rr_ms","rr_std_ms","p_dur_ms","qrs_dur_ms","qtc_ms","sokolow"],
    },
    "incep_ext": {
        "artifact_dir": "inceptiontime_ecg_features_ext_tiffany_flat_nf64",
        "arch": "incep_with_intervals",
        "in_channels": 12,
        "n_interval": 12,
        "intervals_csv": PROJECT_ROOT / "ecg_features_extended.csv",
        "interval_cols": ["rr_ms","rr_std_ms","p_dur_ms","qrs_dur_ms","qtc_ms","sokolow",
                           "sv1_amp","rv5_amp","t_amp_V4_uv","t_amp_V5_uv","s_amp_aVF_uv","p_amp_V1_uv"],
    },
    "multitask": {
        "artifact_dir": "inceptiontime_multitask_tiffany_flat",
        "arch": "incep_multitask",
        "in_channels": 12,
        "n_interval": 6,
        "intervals_csv": None,
        "interval_cols": None,
    },
    "cnn_bilstm": {
        "artifact_dir": "cnn_bilstm_attention_tiffany_flat",
        "arch": "cnn_bilstm",
        "in_channels": 12,
        "n_interval": None,
        "intervals_csv": None,
        "interval_cols": None,
    },
    "cnn_iv": {
        "artifact_dir": "cnn_bilstm_ecg_intervals_tiffany_flat",
        "arch": "cnn_with_intervals",
        "in_channels": 12,
        "n_interval": 6,
        "intervals_csv": PROJECT_ROOT / "ecg_intervals.csv",
        "interval_cols": ["rr_ms","rr_std_ms","p_dur_ms","qrs_dur_ms","qtc_ms","sokolow"],
    },
}

# Ensembles to evaluate with threshold optimization
ENSEMBLE_CONFIGS = [
    ("rib_ext+cnn_iv",                        ["rib_ext", "cnn_iv"]),
    ("rib_ext+incep_iv",                      ["rib_ext", "incep_iv"]),
    ("rib_ext+multitask",                     ["rib_ext", "multitask"]),
    ("rib_ext+multitask+rib_ptbxl_iv+incep_iv",
     ["rib_ext", "multitask", "rib_ptbxl_iv", "incep_iv"]),
    ("rib_ext+incep_ext+multitask+rib_ptbxl_iv",
     ["rib_ext", "incep_ext", "multitask", "rib_ptbxl_iv"]),
]

print("Device:", DEVICE)
print("Models:", list(MODELS_CONFIG.keys()))

In [ ]:
# Shared utilities 

NPZ_RAM_CACHE: dict = {}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def normalize_per_sample_per_lead(signal):
    finite = np.isfinite(signal)
    vc  = np.maximum(finite.sum(axis=0, keepdims=True).astype(np.float32), 1.0)
    mu  = np.where(finite, signal, 0.0).sum(axis=0, keepdims=True).astype(np.float32) / vc
    c   = np.where(finite, signal - mu, 0.0).astype(np.float32)
    std = np.sqrt(np.maximum((c**2).sum(axis=0, keepdims=True)/vc, 1e-6)).astype(np.float32)
    std = np.where(finite.sum(axis=0, keepdims=True)==0, 1.0, std)
    mu  = np.where(finite.sum(axis=0, keepdims=True)==0, 0.0, mu)
    return np.nan_to_num((np.where(finite, signal, mu) - mu) / std,
                         nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def load_npz_arrays(npz_path):
    with np.load(npz_path, allow_pickle=False) as d:
        return (d["signal_matrix"].astype(np.float32),
                d["overlap_mask"].astype(np.uint8),
                d["interpolated_mask"].astype(np.uint8))

def get_npz_arrays(row):
    key = str(row["stem"])
    if key not in NPZ_RAM_CACHE:
        NPZ_RAM_CACHE[key] = load_npz_arrays(row["npz_path"])
    return NPZ_RAM_CACHE[key]

def stable_softmax(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)

def macro_f1_from_probs(y_true, probs, delta=None):
    """Compute macro F1, optionally adding logit offset delta=(d0,d1,d2)."""
    logits = np.log(np.clip(probs, 1e-9, 1.0))
    if delta is not None:
        logits = logits + np.array(delta, dtype=np.float32)
    preds = logits.argmax(axis=1)
    return f1_score(y_true, preds, average="macro", zero_division=0)

def per_class_recall(y_true, probs, delta=None):
    logits = np.log(np.clip(probs, 1e-9, 1.0))
    if delta is not None:
        logits = logits + np.array(delta, dtype=np.float32)
    preds = logits.argmax(axis=1)
    return recall_score(y_true, preds, average=None, zero_division=0, labels=[0,1,2])

def metrics_summary(y_true, probs, delta=None, label=""):
    logits = np.log(np.clip(probs, 1e-9, 1.0))
    if delta is not None:
        logits = logits + np.array(delta, dtype=np.float32)
    preds = logits.argmax(axis=1)
    f1  = f1_score(y_true, preds, average="macro", zero_division=0)
    rec = recall_score(y_true, preds, average=None, zero_division=0, labels=[0,1,2])
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    print(f"{label:50s}  F1={f1:.4f}  AUC={auc:.4f}  Ctrl={rec[0]:.3f}  MR={rec[1]:.3f}  MS={rec[2]:.3f}")
    return f1, auc, rec

print("Utilities loaded.")

In [ ]:
# Dataset classes

class ECGDatasetSimple(Dataset):
    """IN_CHANNELS=12: signal only (overlap zeroed). For InceptionTime, CNN-BiLSTM."""
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
        for _, row in self.frame.iterrows():
            get_npz_arrays(row)  # preload

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        signal, overlap, _ = get_npz_arrays(row)
        signal[overlap.astype(bool)] = 0.0
        sig_ch = normalize_per_sample_per_lead(signal).T  # (12, T)
        label = int(row["class_id"])
        return torch.from_numpy(sig_ch), label, str(row["stem"])


class ECGDatasetMask(Dataset):
    """IN_CHANNELS=36: signal + overlap + interpolated. For Ribeiro models."""
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
        for _, row in self.frame.iterrows():
            get_npz_arrays(row)

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        signal, overlap, interp = get_npz_arrays(row)
        sig_ch  = normalize_per_sample_per_lead(signal).T  # (12, T)
        ovl_ch  = overlap.astype(np.float32).T             # (12, T)
        itp_ch  = interp.astype(np.float32).T              # (12, T)
        x = np.concatenate([sig_ch, ovl_ch, itp_ch], axis=0)  # (36, T)
        label = int(row["class_id"])
        return torch.from_numpy(x), label, str(row["stem"])


class ECGDatasetSimpleWithIntervals(Dataset):
    """IN_CHANNELS=12 + interval vec. For InceptionTime+intervals, CNN+intervals."""
    def __init__(self, frame, interval_lookup, means, stds):
        self.frame   = frame.reset_index(drop=True)
        self.lookup  = interval_lookup
        self.means   = means
        self.stds    = stds
        for _, row in self.frame.iterrows():
            get_npz_arrays(row)

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        signal, overlap, _ = get_npz_arrays(row)
        signal[overlap.astype(bool)] = 0.0
        sig_ch = normalize_per_sample_per_lead(signal).T
        ivec = self.lookup.get(str(row["stem"]), None)
        if ivec is None:
            ivec = self.means.copy()
        else:
            ivec = ivec.copy()
            ivec = np.where(np.isnan(ivec), self.means, ivec)
        ivec = (ivec - self.means) / (self.stds + 1e-8)
        ivec = np.clip(ivec, -5.0, 5.0).astype(np.float32)
        label = int(row["class_id"])
        return torch.from_numpy(sig_ch), torch.from_numpy(ivec), label, str(row["stem"])


class ECGDatasetMaskWithIntervals(Dataset):
    """IN_CHANNELS=36 + interval vec. For Ribeiro+intervals."""
    def __init__(self, frame, interval_lookup, means, stds):
        self.frame   = frame.reset_index(drop=True)
        self.lookup  = interval_lookup
        self.means   = means
        self.stds    = stds
        for _, row in self.frame.iterrows():
            get_npz_arrays(row)

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        signal, overlap, interp = get_npz_arrays(row)
        sig_ch  = normalize_per_sample_per_lead(signal).T
        ovl_ch  = overlap.astype(np.float32).T
        itp_ch  = interp.astype(np.float32).T
        x = np.concatenate([sig_ch, ovl_ch, itp_ch], axis=0)
        ivec = self.lookup.get(str(row["stem"]), None)
        if ivec is None:
            ivec = self.means.copy()
        else:
            ivec = ivec.copy()
            ivec = np.where(np.isnan(ivec), self.means, ivec)
        ivec = (ivec - self.means) / (self.stds + 1e-8)
        ivec = np.clip(ivec, -5.0, 5.0).astype(np.float32)
        label = int(row["class_id"])
        return torch.from_numpy(x), torch.from_numpy(ivec), label, str(row["stem"])


print("Dataset classes defined.")

In [ ]:
# ── Model architectures ──────────────────────────────────────

# ---- Ribeiro ResNet ----
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, kernel_size=17, dropout=0.0):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=p, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.act   = nn.ReLU(inplace=True)
        self.drop  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=p, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.downsample = (
            nn.Sequential(nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False), nn.BatchNorm1d(out_channels))
            if stride != 1 or in_channels != out_channels else None
        )

    def forward(self, x):
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(self.drop(out)))
        return self.act(out + (self.downsample(x) if self.downsample else x))


class ECGResNetWithIntervals(nn.Module):
    def __init__(self, in_channels=36, num_classes=3, n_interval=6, interval_hidden=32, dropout=0.30):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 17, padding=8, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            ResidualBlock1D(64,  64,  1, 17, 0.00), ResidualBlock1D(64,  64,  1, 17, 0.00),
            ResidualBlock1D(64,  128, 2, 17, 0.05), ResidualBlock1D(128, 128, 1, 17, 0.05),
            ResidualBlock1D(128, 196, 2, 17, 0.10), ResidualBlock1D(196, 196, 1, 17, 0.10),
            ResidualBlock1D(196, 256, 2, 17, 0.15), ResidualBlock1D(256, 256, 1, 17, 0.15),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.interval_branch = nn.Sequential(nn.Linear(n_interval, interval_hidden), nn.ReLU(inplace=True))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(256 + interval_hidden, num_classes)

    def forward(self, x, ivec):
        backbone_feat = self.pool(self.blocks(self.stem(x))).squeeze(-1)
        interval_feat = self.interval_branch(ivec)
        return self.head(self.drop(torch.cat([backbone_feat, interval_feat], dim=1)))


# ---- InceptionTime ----
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, n_filters=32, kernel_sizes=(40,20,10), bottleneck=True):
        super().__init__()
        bc = n_filters if bottleneck else in_channels
        self.bottleneck = nn.Conv1d(in_channels, bc, 1, bias=False) if bottleneck else None
        self.convs = nn.ModuleList([nn.Conv1d(bc, n_filters, k, padding=k//2, bias=False) for k in kernel_sizes])
        self.maxpool_conv = nn.Sequential(nn.MaxPool1d(3,1,1), nn.Conv1d(in_channels, n_filters, 1, bias=False))
        out_ch = n_filters * (len(kernel_sizes)+1)
        self.bn = nn.BatchNorm1d(out_ch); self.act = nn.ReLU()

    def forward(self, x):
        bx = self.bottleneck(x) if self.bottleneck is not None else x
        outs = [c(bx) for c in self.convs] + [self.maxpool_conv(x)]
        L = min(o.shape[-1] for o in outs)
        return self.act(self.bn(torch.cat([o[..., :L] for o in outs], dim=1)))


class InceptionTimeWithIntervals(nn.Module):
    def __init__(self, in_channels=12, num_classes=3, n_filters=64,
                 depth=6, kernel_sizes=(40,20,10), n_interval=6, interval_hidden=32, dropout=0.3):
        super().__init__()
        out_ch = n_filters * (len(kernel_sizes)+1)
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        in_ch = in_channels
        for i in range(depth):
            self.blocks.append(InceptionBlock(in_ch, n_filters, kernel_sizes))
            if i % 3 == 0:
                self.shortcuts.append(
                    nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))
                    if in_ch != out_ch else nn.Identity()
                )
            in_ch = out_ch
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.interval_branch = nn.Sequential(nn.Linear(n_interval, interval_hidden), nn.ReLU(inplace=True))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(out_ch + interval_hidden, num_classes)

    def forward(self, x, ivec):
        residual = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i % 3 == 2:
                sc = self.shortcuts[i//3](residual)
                L  = min(x.shape[-1], sc.shape[-1])
                x  = torch.relu(x[..., :L] + sc[..., :L])
                residual = x
        backbone_feat = self.gap(x).squeeze(-1)
        interval_feat = self.interval_branch(ivec)
        return self.head(self.drop(torch.cat([backbone_feat, interval_feat], dim=1)))


class InceptionTimeMultiTask(nn.Module):
    def __init__(self, in_channels=12, num_classes=3, n_filters=64,
                 depth=6, kernel_sizes=(40,20,10), n_interval=6, dropout=0.3):
        super().__init__()
        out_ch = n_filters * (len(kernel_sizes)+1)
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        in_ch = in_channels
        for i in range(depth):
            self.blocks.append(InceptionBlock(in_ch, n_filters, kernel_sizes))
            if i % 3 == 0:
                self.shortcuts.append(
                    nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))
                    if in_ch != out_ch else nn.Identity()
                )
            in_ch = out_ch
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(dropout)
        self.cls_head = nn.Linear(out_ch, num_classes)
        self.reg_head = nn.Linear(out_ch, n_interval)

    def _backbone(self, x):
        residual = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i % 3 == 2:
                sc = self.shortcuts[i//3](residual)
                L  = min(x.shape[-1], sc.shape[-1])
                x  = torch.relu(x[..., :L] + sc[..., :L])
                residual = x
        return self.drop(self.gap(x).squeeze(-1))

    def forward(self, x):
        feat = self._backbone(x)
        return self.cls_head(feat), self.reg_head(feat)


# ---- CNN-BiLSTM ----
class CNNBranch(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, pool=4):
        super().__init__()
        p = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
        )
    def forward(self, x): return self.net(x)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)
    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        return (x * w).sum(dim=1)


class CNNBiLSTMAttention(nn.Module):
    def __init__(self, in_channels=12, num_classes=3, branch_ch=32, lstm_hidden=256, dropout=0.3):
        super().__init__()
        self.branch_s = CNNBranch(in_channels, branch_ch, kernel=7)
        self.branch_m = CNNBranch(in_channels, branch_ch, kernel=15)
        self.branch_l = CNNBranch(in_channels, branch_ch, kernel=31)
        feat_ch = branch_ch * 3
        self.proj = nn.Sequential(nn.Conv1d(feat_ch, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(inplace=True))
        self.bilstm = nn.LSTM(128, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = TemporalAttention(lstm_hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        f_s = self.branch_s(x); f_m = self.branch_m(x); f_l = self.branch_l(x)
        L   = min(f_s.shape[-1], f_m.shape[-1], f_l.shape[-1])
        feat = torch.cat([f_s[..., :L], f_m[..., :L], f_l[..., :L]], dim=1)
        feat = self.proj(feat).transpose(1, 2)
        out, _ = self.bilstm(feat)
        return self.head(self.drop(self.attention(out)))


class CNNBiLSTMWithIntervals(nn.Module):
    def __init__(self, in_channels=12, num_classes=3, branch_ch=32,
                 lstm_hidden=256, n_interval=6, interval_hidden=32, dropout=0.3):
        super().__init__()
        self.branch_s = CNNBranch(in_channels, branch_ch, kernel=7)
        self.branch_m = CNNBranch(in_channels, branch_ch, kernel=15)
        self.branch_l = CNNBranch(in_channels, branch_ch, kernel=31)
        feat_ch = branch_ch * 3
        self.proj = nn.Sequential(nn.Conv1d(feat_ch, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(inplace=True))
        self.bilstm = nn.LSTM(128, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = TemporalAttention(lstm_hidden * 2)
        self.interval_branch = nn.Sequential(nn.Linear(n_interval, interval_hidden), nn.ReLU(inplace=True))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden * 2 + interval_hidden, num_classes)

    def forward(self, x, ivec):
        f_s = self.branch_s(x); f_m = self.branch_m(x); f_l = self.branch_l(x)
        L   = min(f_s.shape[-1], f_m.shape[-1], f_l.shape[-1])
        feat = torch.cat([f_s[..., :L], f_m[..., :L], f_l[..., :L]], dim=1)
        feat = self.proj(feat).transpose(1, 2)
        out, _ = self.bilstm(feat)
        backbone_feat = self.attention(out)
        interval_feat = self.interval_branch(ivec)
        return self.head(self.drop(torch.cat([backbone_feat, interval_feat], dim=1)))


def build_model(arch, in_channels, num_classes, n_interval):
    if arch == "rib_with_intervals":
        return ECGResNetWithIntervals(in_channels, num_classes, n_interval)
    elif arch == "incep_with_intervals":
        return InceptionTimeWithIntervals(in_channels, num_classes, n_interval=n_interval)
    elif arch == "incep_multitask":
        return InceptionTimeMultiTask(in_channels, num_classes, n_interval=n_interval)
    elif arch == "cnn_bilstm":
        return CNNBiLSTMAttention(in_channels, num_classes)
    elif arch == "cnn_with_intervals":
        return CNNBiLSTMWithIntervals(in_channels, num_classes, n_interval=n_interval)
    else:
        raise ValueError(f"Unknown arch: {arch}")

print("Model architectures defined.")

In [ ]:
# Val inference: run per-model, per-fold, save val_logits.npz

@torch.no_grad()
def run_val_inference(model, loader, arch, device):
    model.eval()
    all_y, all_probs, all_stems = [], [], []
    for batch in loader:
        if arch in ("cnn_bilstm", "incep_multitask"):  # no interval input
            x, y, stems = batch
            x = x.to(device)
            if arch == "incep_multitask":
                logits, _ = model(x)  # cls_head only
            else:
                logits = model(x)
        else:  # with intervals
            x, ivec, y, stems = batch
            x, ivec = x.to(device), ivec.to(device)
            logits = model(x, ivec)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_y.extend(y.tolist() if hasattr(y, 'tolist') else list(y))
        all_probs.append(probs)
        all_stems.extend(list(stems))
    return np.array(all_y), np.concatenate(all_probs), np.array(all_stems)


def make_val_dataset(val_df, cfg):
    arch = cfg["arch"]
    if arch in ("cnn_bilstm", "incep_multitask"):
        return ECGDatasetSimple(val_df)
    elif arch in ("incep_with_intervals", "cnn_with_intervals"):
        intervals_df = pd.read_csv(cfg["intervals_csv"])
        lookup = dict(zip(intervals_df["stem"].astype(str),
                         intervals_df[cfg["interval_cols"]].values.astype(np.float32)))
        # means/stds loaded from checkpoint below
        return None, lookup  # handled in main loop
    elif arch == "rib_with_intervals":
        intervals_df = pd.read_csv(cfg["intervals_csv"])
        lookup = dict(zip(intervals_df["stem"].astype(str),
                         intervals_df[cfg["interval_cols"]].values.astype(np.float32)))
        return None, lookup


def run_all_val_inference(models_to_run=None):
    if models_to_run is None:
        models_to_run = list(MODELS_CONFIG.keys())

    for model_name in models_to_run:
        cfg = MODELS_CONFIG[model_name]
        artifact_dir = ARTIFACT_ROOT / cfg["artifact_dir"]
        arch = cfg["arch"]
        print(f"\n{'='*60}")
        print(f"Model: {model_name} ({arch})")

        # Load interval lookup if needed
        interval_lookup = None
        if cfg["intervals_csv"] is not None:
            intervals_df = pd.read_csv(cfg["intervals_csv"])
            interval_lookup = dict(zip(
                intervals_df["stem"].astype(str),
                intervals_df[cfg["interval_cols"]].values.astype(np.float32)
            ))
            print(f"  Intervals loaded: {len(interval_lookup)} stems")

        for fold in range(NUM_FOLDS):
            fold_dir     = artifact_dir / f"fold_{fold}"
            val_logits_p = fold_dir / "val_logits.npz"

            if val_logits_p.exists():
                print(f"  Fold {fold}: val_logits.npz exists — skipping.")
                continue

            ckpt_path = fold_dir / "best_model.pt"
            val_csv   = fold_dir / "val_split.csv"
            if not ckpt_path.exists() or not val_csv.exists():
                print(f"  Fold {fold}: missing checkpoint or val_split — skip.")
                continue

            val_df = pd.read_csv(val_csv)

            # Load checkpoint
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            model = build_model(arch, cfg["in_channels"], len(CLASS_NAMES), cfg["n_interval"])
            model.load_state_dict(ckpt["model_state_dict"])
            model.to(DEVICE).eval()

            # Build val DataLoader
            if arch in ("cnn_bilstm", "incep_multitask"):
                dataset = ECGDatasetSimple(val_df)
                loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            elif arch in ("incep_with_intervals", "cnn_with_intervals"):
                means = ckpt["interval_means"]
                stds  = ckpt["interval_stds"]
                dataset = ECGDatasetSimpleWithIntervals(val_df, interval_lookup, means, stds)
                loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            elif arch == "rib_with_intervals":
                means = ckpt["interval_means"]
                stds  = ckpt["interval_stds"]
                dataset = ECGDatasetMaskWithIntervals(val_df, interval_lookup, means, stds)
                loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            y_true, probs, stems = run_val_inference(model, loader, arch, DEVICE)
            np.savez(val_logits_p, y_true=y_true, probs=probs, stems=stems)

            f1 = f1_score(y_true, probs.argmax(axis=1), average="macro", zero_division=0)
            print(f"  Fold {fold}: val F1={f1:.4f} ({len(y_true)} samples) — saved.")

            del model
            torch.cuda.empty_cache()


run_all_val_inference()

In [ ]:
# Load OOF val predictions & heldout predictions

def load_oof_val(model_name):
    """Load and pool val predictions across all 5 folds (OOF)."""
    cfg = MODELS_CONFIG[model_name]
    artifact_dir = ARTIFACT_ROOT / cfg["artifact_dir"]
    all_y, all_probs, all_stems = [], [], []
    for fold in range(NUM_FOLDS):
        p = artifact_dir / f"fold_{fold}" / "val_logits.npz"
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")
        d = np.load(p)
        all_y.append(d["y_true"])
        all_probs.append(d["probs"])
        all_stems.append(d["stems"])
    return np.concatenate(all_y), np.concatenate(all_probs), np.concatenate(all_stems)


def load_heldout(model_name):
    """Load and average heldout predictions across all 5 folds."""
    cfg = MODELS_CONFIG[model_name]
    artifact_dir = ARTIFACT_ROOT / cfg["artifact_dir"]
    probs_list = []
    y_true = None
    for fold in range(NUM_FOLDS):
        p = artifact_dir / f"fold_{fold}" / "heldout_logits.npz"
        d = np.load(p)
        probs_list.append(d["probs"])
        if y_true is None:
            y_true = d["y_true"]
    return y_true, np.mean(probs_list, axis=0)


def build_ensemble_oof(model_names):
    """
    Build OOF ensemble val predictions.
    Since different models may have slightly different val samples (due to interval filtering),
    we align by stem and take the intersection.
    """
    per_model = {}
    for name in model_names:
        y, probs, stems = load_oof_val(name)
        per_model[name] = {s: (p, yi) for s, p, yi in zip(stems, probs, y)}

    # Intersection of stems
    common_stems = set(per_model[model_names[0]].keys())
    for name in model_names[1:]:
        common_stems &= set(per_model[name].keys())
    common_stems = sorted(common_stems)
    print(f"  Ensemble OOF: {len(common_stems)} common stems from {model_names}")

    y_true = np.array([per_model[model_names[0]][s][1] for s in common_stems])
    avg_probs = np.mean(
        [np.array([per_model[name][s][0] for s in common_stems]) for name in model_names],
        axis=0
    )
    return y_true, avg_probs


def build_ensemble_heldout(model_names):
    """Average heldout predictions across all models."""
    probs_list = []
    y_true = None
    for name in model_names:
        yt, probs = load_heldout(name)
        probs_list.append(probs)
        if y_true is None:
            y_true = yt
    return y_true, np.mean(probs_list, axis=0)


print("Data loading functions defined.")

In [ ]:
# ── Threshold grid search ────────────────────────────────────
# Fairness guarantee: delta is tuned only on OOF val predictions.
# Heldout set is NEVER used for threshold selection.

def find_best_delta(y_oof, probs_oof, grid_steps=41, search_range=1.5):
    """
    Grid search for optimal logit offset delta = [0, d1, d2].
    d0=0 (fixed as reference), d1/d2 tuned to boost MR/MS recall.
    Objective: maximize macro F1 on OOF val predictions.
    """
    log_probs = np.log(np.clip(probs_oof, 1e-9, 1.0))  # (N, 3)

    grid = np.linspace(-search_range, search_range, grid_steps)
    best_f1, best_delta = -1.0, (0.0, 0.0, 0.0)

    for d1 in grid:
        for d2 in grid:
            delta = np.array([0.0, d1, d2], dtype=np.float32)
            preds = (log_probs + delta).argmax(axis=1)
            f1 = f1_score(y_oof, preds, average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1, best_delta = f1, (0.0, float(d1), float(d2))

    return best_delta, best_f1


def evaluate_threshold(ensemble_name, model_names):
    print(f"\n{'─'*60}")
    print(f"Ensemble: {ensemble_name}")

    # OOF val predictions (for tuning)
    y_oof, probs_oof = build_ensemble_oof(model_names)

    # Heldout predictions (for final reporting)
    y_held, probs_held = build_ensemble_heldout(model_names)

    # Baseline (no threshold tuning)
    print("  Baseline (delta=0):")
    metrics_summary(y_held, probs_held, delta=None, label="    Heldout (untuned)")

    print("  OOF val F1 (untuned):", f"{macro_f1_from_probs(y_oof, probs_oof):.4f}")

    # Find best delta on OOF val
    best_delta, best_oof_f1 = find_best_delta(y_oof, probs_oof)
    print(f"  Best delta (tuned on OOF val): {best_delta}  OOF F1={best_oof_f1:.4f}")

    # Apply to heldout
    print("  With threshold optimization:")
    f1_tuned, auc_tuned, rec_tuned = metrics_summary(
        y_held, probs_held, delta=best_delta, label="    Heldout (tuned)  ")

    f1_base = macro_f1_from_probs(y_held, probs_held)
    print(f"  Delta F1: {f1_tuned - f1_base:+.4f}  ({f1_base:.4f} → {f1_tuned:.4f})")

    return {
        "ensemble": ensemble_name,
        "f1_base": f1_base,
        "f1_tuned": f1_tuned,
        "delta_f1": f1_tuned - f1_base,
        "best_delta": best_delta,
        "oof_f1_before": macro_f1_from_probs(y_oof, probs_oof),
        "oof_f1_after": best_oof_f1,
        "auc_tuned": auc_tuned,
        "rec_ctrl_tuned": rec_tuned[0],
        "rec_mr_tuned": rec_tuned[1],
        "rec_ms_tuned": rec_tuned[2],
    }


print("Threshold functions defined.")

In [ ]:
# ── Run threshold optimization for all ensembles ─────────────

all_results = []
for ens_name, model_names in ENSEMBLE_CONFIGS:
    result = evaluate_threshold(ens_name, model_names)
    all_results.append(result)

# Also evaluate single best model (rib_ext) for reference
print("\n" + "─"*60)
print("Single model: rib_ext")
y_oof_rib, probs_oof_rib = load_oof_val("rib_ext")
y_held_rib, probs_held_rib = load_heldout("rib_ext")
print("  OOF val F1 (untuned):", f"{macro_f1_from_probs(y_oof_rib, probs_oof_rib):.4f}")
best_delta_rib, best_oof_rib = find_best_delta(y_oof_rib, probs_oof_rib)
print(f"  Best delta: {best_delta_rib}  OOF F1={best_oof_rib:.4f}")
f1_base_rib = macro_f1_from_probs(y_held_rib, probs_held_rib)
f1_tuned_rib, _, rec_rib = metrics_summary(y_held_rib, probs_held_rib, best_delta_rib, "  rib_ext (tuned)")
print(f"  Delta F1: {f1_tuned_rib - f1_base_rib:+.4f}  ({f1_base_rib:.4f} → {f1_tuned_rib:.4f})")

all_results.append({
    "ensemble": "rib_ext (single)",
    "f1_base": f1_base_rib, "f1_tuned": f1_tuned_rib,
    "delta_f1": f1_tuned_rib - f1_base_rib,
    "best_delta": best_delta_rib,
    "oof_f1_before": macro_f1_from_probs(y_oof_rib, probs_oof_rib),
    "oof_f1_after": best_oof_rib,
    "auc_tuned": float("nan"),
    "rec_ctrl_tuned": rec_rib[0], "rec_mr_tuned": rec_rib[1], "rec_ms_tuned": rec_rib[2],
})

In [ ]:
# ── Summary report ───────────────────────────────────────────

df = pd.DataFrame(all_results)
df = df.sort_values("f1_tuned", ascending=False)

print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION RESULTS")
print("="*80)
print(f"{'Ensemble':<45} {'F1 Base':>8} {'F1 Tuned':>9} {'Delta':>7} {'Delta'}",)
print(f"{'':45} {'':8} {'':9} {'Ctrl':>6} {'MR':>6} {'MS':>6}")
print("-"*80)
for _, row in df.iterrows():
    print(f"{row['ensemble']:<45} {row['f1_base']:>8.4f} {row['f1_tuned']:>9.4f}",
          f"  Ctrl={row['rec_ctrl_tuned']:.3f} MR={row['rec_mr_tuned']:.3f} MS={row['rec_ms_tuned']:.3f}",
          f"  delta=({row['best_delta'][0]:.2f},{row['best_delta'][1]:.2f},{row['best_delta'][2]:.2f})")

df.to_csv("dl_artifacts/threshold_optimization_results.csv", index=False)
print("\nSaved: dl_artifacts/threshold_optimization_results.csv")